In [ ]:
###
    # This code calculates the moisture-driven sensitivity of soil temperature on seasonal scale.
    # 1. Read soil temperature data includes (1)offsup:"../Data/Moisture_driven/output/*_moisture_driven_soil_Ts_offsup.nc" (2)onsup:"../Data/Moisture_driven/output/*_moisture_driven_soil_Ts_onsup.nc" (calculated from Mositure_dirven model)
    # 2. Select soil temperature data from 50-100 years at depths of 1.6m, 2.4m and 3.2m.
    # 3. Calculate the seasonal soil temperature sensitivity of each grid in the permafrost region across northern Eurasia.
    # 4. Calculate the Multi-year average of seasonal sensitivity of each grid.
    # 5. Calculate the area-weighted spatial average of all grids.
    # 6. Output: "../Data/Moisture_driven/Ts_seasonal_sensitivity*.csv", as Figure 4d and the x-axis of Figure 4a-c. 
###

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import glob
from scipy import stats
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ===============================
# Area weighting function
# ===============================

def area_weight_mean(data2D, lat, lon):

    rad = np.pi/180
    latr = lat*rad

    re = 6371220.0

    dlon = np.abs(lon[1]-lon[0])*rad
    dlat = np.abs(lat[1]-lat[0])*rad

    sin1 = np.sin(latr+dlat/2)
    sin2 = np.sin(latr-dlat/2)

    area = (re**2)*dlon*(sin1-sin2)[:,None]

    mask = ~np.isnan(data2D)

    ws = np.nansum(data2D*area*mask)
    tw = np.nansum(area*mask)

    return np.nan if tw==0 else ws/tw

In [ ]:
depths = ['1.6m', '2.4m', '3.2m']
models = ["CESM2","CESM2-FV2","CESM2-WACCM",
          "CNRM-CM6-1","CNRM-ESM2-1",
          "GFDL-ESM4","NorESM2-LM",
          "NorESM2-MM","TaiESM1"
          ]

# ===============================
# Parameters
# ===============================
start_year = 50                                 # Start from 50th year
nyear = 100-start_year
mon_per_year = 12

# Soil depth
nl_soil = 15                                    # upper bound of array
x_array = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15])
z_src = 0.025*(np.exp(0.5*(x_array-0.5))-1)     # node depth [m]
z_tgt = np.arange(0,3.3,0.1)                    # 0–3.2m

idx_02 = np.argmin(np.abs(z_tgt-0.2))
idx_16 = np.argmin(np.abs(z_tgt-1.6))
idx_24 = np.argmin(np.abs(z_tgt-2.4))
idx_32 = np.argmin(np.abs(z_tgt-3.2))

# ===============================
# Read permafrost extent mask
# ===============================
q16 = xr.open_dataset('../Data/probability_lt_threshold_160.nc')
q24 = xr.open_dataset('../Data/probability_lt_threshold_240.nc')
q32 = xr.open_dataset('../Data/probability_lt_threshold_320.nc')

per16 = q16['probability_lt_threshold'][:,:360]
per24 = q24['probability_lt_threshold'][:,:360]
per32 = q32['probability_lt_threshold'][:,:360]

In [ ]:
amplitude_ratio_onsup = pd.DataFrame(index=depths, columns=models)
for model in models:

    # ===============================
    # Read data
    # ===============================
    ncfile = model+"_moisture_driven_soil_Ts_onsup.nc"
    ds = xr.open_dataset(ncfile)

    T = ds["tsoil"].values      # (time,layer,lat,lon)
    lat = ds["lat"].values
    lon = ds["lon"].values

    # Bilinear interpolation to model grid
    per16_interp = per16.interp(lat=lat, lon=lon, method="linear")
    per24_interp = per24.interp(lat=lat, lon=lon, method="linear")
    per32_interp = per32.interp(lat=lat, lon=lon, method="linear")

    per16_interp = per16_interp.values
    per24_interp = per24_interp.values
    per32_interp = per32_interp.values

    # ===============================
    # 1. Extract the last 50 years
    # ===============================

    ist = start_year*12
    ied = ist+nyear*12

    T_600 = T[ist:ied]     # (600,z,lat,lon)
    ntime,nlayer,nlat,nlon = T_600.shape

    # ===============================
    # 2. Vertical interpolation
    # ===============================
    
    T_int = np.full(
        (ntime,len(z_tgt),nlat,nlon),
        np.nan
    )

    for k in range(nlat):
        for l in range(nlon):

            prof = T_600[:,:,k,l]   # (time,layer)

            if np.all(np.isnan(prof)):
                continue
            for i in range(ntime):
                T_int[i,:,k,l] = np.interp(
                    z_tgt,
                    z_src,
                    prof[i,:],
                    left = prof[i,0],
                    right = prof[i,-1]
                ).T
            # (600,z,lat,lon)

    # ===============================
    # 3. reshape → (year,mon,z,lat,lon)
    # ===============================

    T3 = T_int.reshape(
        nyear,12,
        len(z_tgt),nlat,nlon
    )

    # ===============================
    # 4. Compute amplitude
    # ===============================

    amp = np.nanmax(T3,axis=1)-np.nanmin(T3,axis=1)
    # (year,z,lat,lon)

    mean_amp = np.nanmean(amp,axis=0)
    # (z,lat,lon)

    # ===============================
    # 5. Ratio
    # ===============================

    R16 = mean_amp[idx_16]/mean_amp[idx_02]
    R24 = mean_amp[idx_24]/mean_amp[idx_02]
    R32 = mean_amp[idx_32]/mean_amp[idx_02]
    # Filter by permafrost extent
    R16 = np.where(per16_interp!=0, R16, np.nan)
    R24 = np.where(per24_interp!=0, R24, np.nan)
    R32 = np.where(per32_interp!=0, R32, np.nan)

    # ===============================
    # 6. Area weighting
    # ===============================

    m16 = area_weight_mean(R16,lat,lon)
    m24 = area_weight_mean(R24,lat,lon)
    m32 = area_weight_mean(R32,lat,lon)

    amplitude_ratio_onsup[model]['1.6m'] = m16
    amplitude_ratio_onsup[model]['2.4m'] = m24
    amplitude_ratio_onsup[model]['3.2m'] = m32
amplitude_ratio_onsup.to_csv("../Data/Moisture_driven/Ts_seasonal_sensitivity_onsup.csv")

In [ ]:
amplitude_ratio_offsup = pd.DataFrame(index=depths, columns=models)
for model in models:
    # ===============================
    # Read data
    # ===============================
    ncfile = model+"_moisture_driven_soil_Ts_offsup.nc"
    ds = xr.open_dataset(ncfile)

    T = ds["tsoil"].values      # (time,layer,lat,lon)
    lat = ds["lat"].values
    lon = ds["lon"].values

    # Bilinear interpolation to model grid
    per16_interp = per16.interp(lat=lat, lon=lon, method="linear")
    per24_interp = per24.interp(lat=lat, lon=lon, method="linear")
    per32_interp = per32.interp(lat=lat, lon=lon, method="linear")

    per16_interp = per16_interp.values
    per24_interp = per24_interp.values
    per32_interp = per32_interp.values
    # ===============================
    # 1. Extract the last 50 years
    # ===============================
    ist = start_year*12
    ied = ist+nyear*12

    T_600 = T[ist:ied]     # (600,z,lat,lon)
    ntime,nlayer,nlat,nlon = T_600.shape

    # ===============================
    # 2. Vertical interpolation
    # ===============================

    T_int = np.full(
        (ntime,len(z_tgt),nlat,nlon),
        np.nan
    )

    for k in range(nlat):
        for l in range(nlon):

            prof = T_600[:,:,k,l]   # (time,layer)

            if np.all(np.isnan(prof)):
                continue
            for i in range(ntime):
                T_int[i,:,k,l] = np.interp(
                    z_tgt,
                    z_src,
                    prof[i,:],
                    left = prof[i,0],
                    right = prof[i,-1]
                ).T
            # (600,z,lat,lon)

    # ===============================
    # 3. reshape → (year,mon,z,lat,lon)
    # ===============================

    T3 = T_int.reshape(
        nyear,12,
        len(z_tgt),nlat,nlon
    )

    # ===============================
    # 4. Compute amplitude
    # ===============================

    amp = np.nanmax(T3,axis=1)-np.nanmin(T3,axis=1)
    # (year,z,lat,lon)

    mean_amp = np.nanmean(amp,axis=0)
    # (z,lat,lon)

    # ===============================
    # 5. Ratio
    # ===============================

    R16 = mean_amp[idx_16]/mean_amp[idx_02]
    R24 = mean_amp[idx_24]/mean_amp[idx_02]
    R32 = mean_amp[idx_32]/mean_amp[idx_02]
    # Filter by permafrost extent
    R16 = np.where(per16_interp!=0, R16, np.nan)
    R24 = np.where(per24_interp!=0, R24, np.nan)
    R32 = np.where(per32_interp!=0, R32, np.nan)

    # ===============================
    # 6. area_weighting
    # ===============================

    m16 = area_weight_mean(R16,lat,lon)
    m24 = area_weight_mean(R24,lat,lon)
    m32 = area_weight_mean(R32,lat,lon)

    amplitude_ratio_offsup[model]['1.6m'] = m16
    amplitude_ratio_offsup[model]['2.4m'] = m24
    amplitude_ratio_offsup[model]['3.2m'] = m32
amplitude_ratio_offsup.to_csv("../Data/Moisture_driven/Ts_seasonal_sensitivity_offsup.csv")